# Mã Autokey (Autokey Cipher)

## Giới thiệu

Mã Autokey (Autokey Cipher) là một phương pháp mã hóa thuộc nhóm
mã thay thế đa ký tự (polyalphabetic substitution cipher).

Khác với mã thay thế đơn ký tự, trong Autokey Cipher mỗi ký tự
plaintext có thể được mã hóa thành các ký tự khác nhau tùy thuộc
vào khóa đang sử dụng.

Khóa của Autokey Cipher gồm:

- một khóa ban đầu
- phần còn lại của khóa được tạo từ chính plaintext.

## 1. Cơ sở lý thuyết

#### Giả sử:

P = P₁ P₂ P₃ ... (plaintext)

C = C₁ C₂ C₃ ... (ciphertext)

K = (k₁, P₁, P₂, P₃, ...)

#### Công thức mã hóa

Cᵢ = (Pᵢ + kᵢ) mod 26

#### Công thức giải mã

Pᵢ = (Cᵢ − kᵢ) mod 26

## 2. Ý tưởng thuật toán

#### Thuật toán mã hóa

1. Chuyển plaintext thành số từ 0–25.
2. Chọn khóa ban đầu k₁.
3. Tạo keystream:

   k₁, P₁, P₂, P₃, ...

4. Với mỗi ký tự:

   Cᵢ = (Pᵢ + kᵢ) mod 26

5. Chuyển kết quả trở lại ký tự.

#### Thuật toán giải mã

1. Lấy khóa ban đầu k₁.
2. Với ký tự đầu tiên:

   P₁ = (C₁ − k₁) mod 26

3. Sau khi tìm được P₁, dùng nó làm khóa tiếp theo.

4. Lặp lại cho các ký tự còn lại.

## 3. Cài đặt thuật toán mã hóa

In [1]:
def autokey_encrypt(text, key):
    result = ""

    key_stream = [key]

    for ch in text[:-1]:
        if ch.isalpha():
            key_stream.append(ord(ch.lower()) - ord('a'))

    i = 0

    for ch in text:

        if ch.isalpha():
            p = ord(ch.lower()) - ord('a')
            k = key_stream[i]

            c = (p + k) % 26
            if ch.islower():
                result += chr(c + ord('a'))
            else:
                result += chr(c + ord('A'))
            i += 1
        else:
            result += ch

    return result

## 4. Cài đặt thuật toán giải mã

In [2]:
def autokey_decrypt(cipher, key):
    result = ""

    key_stream = [key]

    i = 0

    for ch in cipher:

        if ch.isalpha():

            c = ord(ch.lower()) - ord('a')
            k = key_stream[i]

            p = (c - k) % 26
            if ch.islower():
                plain_char = chr(p + ord('a'))
            else:
                plain_char = chr(p + ord('A'))
            result += plain_char

            key_stream.append(p)

            i += 1

        else:
            result += ch

    return result

## 5. Giải thích thuật toán

#### Hàm mã hóa

Hàm `autokey_encrypt()` thực hiện các bước sau:

1. Chuyển toàn bộ văn bản về chữ thường để thuận tiện xử lý.
2. Khởi tạo **keystream** với khóa ban đầu `k₁`.
3. Các phần tử tiếp theo của keystream được lấy từ chính plaintext
   (theo đúng nguyên lý của Autokey Cipher).
4. Với mỗi ký tự:
   - chuyển ký tự sang số từ 0–25 bằng `ord(ch) - ord('a')`
   - lấy giá trị khóa tương ứng trong keystream
   - tính:

   C = (P + K) mod 26

5. Kết quả được chuyển lại thành ký tự bằng `chr()`.

#### Hàm giải mã

Hàm `autokey_decrypt()` thực hiện quá trình ngược lại:

1. Khởi tạo keystream bằng khóa ban đầu `k₁`.
2. Với mỗi ký tự ciphertext:
   - chuyển ký tự sang giá trị số
   - áp dụng công thức:

   P = (C − K) mod 26

3. Sau khi tìm được ký tự plaintext, giá trị của nó sẽ được
   **thêm vào keystream** để dùng cho bước tiếp theo.

Nhờ đó, keystream được xây dựng dần dần từ plaintext
trong quá trình giải mã.

## 6. Chạy thử chương trình

In [4]:
text = input("Hay nhap dong text muon ma hoa:")
key = int(input("Hay nhap gia tri khoa ban dau (initial key):"))

keystream = [key]
keystream_with_value = [key]

for t in text[:-1]:
    keystream.append(t)
    keystream_with_value.append(ord(t.lower()) - ord('a'))
    
print("Keystream ban dau:", keystream)
print("Keystream sau bien doi:", keystream_with_value)

cipher = autokey_encrypt(text, key)
print("Dong text sau khi ma hoa:", cipher)

plain = autokey_decrypt(cipher, key)
print("Dong text sau khi giai ma:", plain)

Hay nhap dong text muon ma hoa: HUST
Hay nhap gia tri khoa ban dau (initial key): 12


Keystream ban dau: [12, 'H', 'U', 'S']
Keystream sau bien doi: [12, 7, 20, 18]
Dong text sau khi ma hoa: TBML
Dong text sau khi giai ma: HUST


## 7. Minh họa quá trình mã hóa

Plaintext: **HUST**  
Khóa ban đầu: **k₁ = 12**

Quy ước: A=0, B=1, ..., Z=25

Keystream: **12, H, U, S**

| Plaintext | H | U | S | T |
|-----------|---|---|---|---|
| P (giá trị) | 7 | 20 | 18 | 19 |
| Key (K) | 12 | 7 | 20 | 18 |
| Cipher (C) = (P + K) mod 26 | 19 | 1 | 12 | 11 |
| Ciphertext | T | B | M | L |

Ciphertext: **TBML**

## 8. Minh họa quá trình giải mã

Ciphertext: **TBML**  
Khóa ban đầu: **k₁ = 12**

| Ciphertext | T | B | M | L |
|------------|---|---|---|---|
| C (giá trị) | 19 | 1 | 12 | 11 |
| Key (K) | 12 | 7 | 20 | 18 |
| Plain (P) = (C − K) mod 26 | 7 | 20 | 18 | 19 |
| Plaintext | H | U | S | T |

Plaintext: **HUST**